# LAB1 - Bài 8 (Titanic): Tạo thuộc tính mới (age_class, fare_per_person)

**Họ tên:** Đinh Công Vinh Quang  
**MSSV:** 6451071061

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

HO_TEN, MSSV = "Đinh Công Vinh Quang", "6451071061"
print(f"{HO_TEN} - {MSSV}")

Đinh Công Vinh Quang - 6451071061


In [2]:
df = pd.read_csv("titanic.csv")

In [3]:
# --- Tiền xử lý (Câu 6) ---
df["deck"] = df["cabin"].str[0]                       # giữ lại ký tự đầu của cabin (boong) trước khi xóa
df = df.drop(columns=["body", "cabin", "boat", "home.dest", "ticket"])

# tuổi: sinh ngẫu nhiên trong [mean - std, mean + std]
rng = np.random.default_rng(42)
mean, std = df["age"].mean(), df["age"].std()
n_null = df["age"].isnull().sum()
df.loc[df["age"].isnull(), "age"] = rng.integers(int(mean - std), int(mean + std) + 1, n_null)

# cổng lên tàu: điền giá trị phổ biến nhất (mode); fare thiếu 1 giá trị: điền trung vị
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"] = df["fare"].fillna(df["fare"].median())
df["deck"] = df["deck"].fillna("U")                   # U = Unknown

In [4]:
# --- Biến đổi thuộc tính (Câu 7) ---
# chia nhóm fare (tính trên giá trị thực, trước khi ép kiểu nguyên)
df["fare_group"] = pd.cut(df["fare"], bins=[-0.01, 7.91, 14.454, 31, 99, 250, np.inf], labels=range(1, 7))
df["fare_group_qcut"] = pd.qcut(df["fare"].rank(method="first"), 6, labels=range(1, 7))   # chia cân bằng bằng qcut

df["fare"] = df["fare"].astype(int)
df["age"] = df["age"].astype(int)

df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
df["title"] = df["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
common = ["Mr", "Miss", "Mrs", "Master"]
df["title"] = df["title"].where(df["title"].isin(common), "Rare")

df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"S": 0, "C": 1, "Q": 2})

df["age_group"] = pd.cut(df["age"], bins=[-1, 10, 18, 22, 27, 33, 40, 66, 200], labels=range(1, 9))

In [5]:
# --- Thuộc tính mới (Câu 8) ---
df["family_size"] = df["sibsp"] + df["parch"]
df["age_class"] = df["age"] * df["pclass"]
df["fare_per_person"] = (df["fare"] / (df["family_size"] + 1)).astype(int)

In [6]:
df[["name", "age", "pclass", "age_class", "fare", "family_size", "fare_per_person"]].head(10)

,name,age,pclass,age_class,fare,family_size,fare_per_person
0,"Allen, Miss. Elisabeth Walton",29,1,29,211,0,211
1,"Allison, Master. Hudson Trevor",0,1,0,151,3,37
2,"Allison, Miss. Helen Loraine",2,1,2,151,3,37
3,"Allison, Mr. Hudson Joshua Creighton",30,1,30,151,3,37
4,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",25,1,25,151,3,37
5,"Anderson, Mr. Harry",48,1,48,26,0,26
6,"Andrews, Miss. Kornelia Theodosia",63,1,63,77,1,38
7,"Andrews, Mr. Thomas Jr",39,1,39,0,0,0
8,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",53,1,53,51,2,17
9,"Artagaveytia, Mr. Ramon",71,1,71,49,0,49


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   pclass           1309 non-null   int64   
 1   survived         1309 non-null   int64   
 2   name             1309 non-null   str     
 3   sex              1309 non-null   int64   
 4   age              1309 non-null   int64   
 5   sibsp            1309 non-null   int64   
 6   parch            1309 non-null   int64   
 7   fare             1309 non-null   int64   
 8   embarked         1309 non-null   int64   
 9   deck             1309 non-null   str     
 10  fare_group       1309 non-null   category
 11  fare_group_qcut  1309 non-null   category
 12  title            1309 non-null   str     
 13  age_group        1309 non-null   category
 14  family_size      1309 non-null   int64   
 15  age_class        1309 non-null   int64   
 16  fare_per_person  1309 non-null   int64   
dtypes: cat